In [ ]:
# ── CELL 0: Imports + Dataset Setup ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

n = 500
df = pd.DataFrame({
    'employee_id':  range(1, n + 1),
    'age':          np.random.randint(22, 65, n),
    'salary':       np.random.exponential(60000, n),
    'experience':   np.random.randint(0, 40, n),
    'score':        np.random.uniform(40, 100, n).round(1),
    'department':   np.random.choice(['Eng','Sales','HR','Product','Legal'], n),
    'gender':       np.random.choice(['M','F','M','M'], n),
    'promoted':     np.random.choice([0, 1], n, p=[0.75, 0.25])
})
# Introduce realistic messiness
df.loc[df.sample(frac=0.08, random_state=1).index, 'salary']     = np.nan
df.loc[df.sample(frac=0.03, random_state=2).index, 'score']      = np.nan
df.loc[df.sample(frac=0.02, random_state=3).index, 'department'] = np.nan
df = pd.concat([df, df.sample(15, random_state=4)], ignore_index=True)
df.loc[df.sample(frac=0.01, random_state=5).index, 'age'] = 999

print('Raw dataset shape:', df.shape)
print(df.head())

In [ ]:
# ── CELL 1: STEP 1 — Structure and Types ──────────────────────────────────────
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nFirst 5 rows:')
print(df.head())
print('\nRandom sample:')
print(df.sample(5))

In [ ]:
# ── CELL 2: STEP 2 — Missing Values ───────────────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = df.isnull().mean() * 100

missing_df = pd.DataFrame({
    'count': missing_count,
    'pct':   missing_pct.round(2)
}).sort_values('pct', ascending=False)

print('Missing Values:')
print(missing_df[missing_df['count'] > 0])

fig, ax = plt.subplots(figsize=(8, 4))
cols_with_missing = missing_pct[missing_pct > 0].sort_values()
cols_with_missing.plot(kind='barh', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Missing Value Percentage by Column')
ax.set_xlabel('% Missing')
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 3: STEP 3 — Duplicates ───────────────────────────────────────────────
print('Shape before:', df.shape)
print('Duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
df = df.reset_index(drop=True)

print('Shape after removing duplicates:', df.shape)

In [ ]:
# ── CELL 4: STEP 4 — Distributions ────────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('employee_id').tolist()

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))

for i, col in enumerate(numeric_cols):
    ax = axes.flatten()[i]
    sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax, color=f'C{i}')
    ax.set_title(f'{col} | skew={df[col].skew():.2f}')

for j in range(i + 1, n_rows * n_cols):
    axes.flatten()[j].set_visible(False)

plt.suptitle('Feature Distributions (skew shown in title)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nSkewness scores:')
print(df[numeric_cols].skew().sort_values(ascending=False).round(2))

In [ ]:
# ── CELL 5: STEP 5 — Outlier Detection ────────────────────────────────────────
def detect_outliers_iqr(series):
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

print('Outlier Summary:')
for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df[col].dropna())
    if len(outliers) > 0:
        print(f'  {col}: {len(outliers)} outliers | range [{lower:.1f}, {upper:.1f}]')

fig, axes = plt.subplots(1, min(3, len(numeric_cols)), figsize=(14, 4))
for i, col in enumerate(numeric_cols[:3]):
    sns.boxplot(y=df[col].dropna(), ax=axes[i], color='steelblue')
    axes[i].set_title(col)
plt.suptitle('Outlier Detection via Box Plots', fontsize=13)
plt.tight_layout()
plt.show()

# Fix age outliers (age=999 is clearly a data error)
print('\nAge outliers (age=999):', (df['age'] == 999).sum())
df = df[df['age'] < 100]
df = df.reset_index(drop=True)
print('Shape after fixing age outliers:', df.shape)

In [ ]:
# ── CELL 6: Log Transform Skewed Features ─────────────────────────────────────
print('Salary skew BEFORE:', df['salary'].skew().round(2))
df['salary_log'] = np.log1p(df['salary'])
print('Salary skew AFTER log:', df['salary_log'].skew().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['salary'].dropna(), bins=40, kde=True, ax=axes[0], color='tomato')
axes[0].set_title(f'Salary BEFORE | skew={df["salary"].skew():.2f}')

sns.histplot(df['salary_log'].dropna(), bins=40, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title(f'Salary AFTER log1p | skew={df["salary_log"].skew():.2f}')

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 7: STEP 6 — Correlation Analysis ─────────────────────────────────────
corr_cols = ['age', 'salary_log', 'experience', 'score', 'promoted']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5,
    mask=np.triu(np.ones_like(corr, dtype=bool)),
    ax=ax
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelation with target (promoted):')
target_corr = corr['promoted'].drop('promoted').sort_values(key=abs, ascending=False)
print(target_corr.round(3))

In [ ]:
# ── CELL 8: STEP 7 — Target Analysis ──────────────────────────────────────────
target = 'promoted'

print('Target distribution:')
print(df[target].value_counts())
print('\nPercentage:')
print((df[target].value_counts(normalize=True) * 100).round(1))

ratio = df[target].value_counts().min() / df[target].value_counts().max()
print(f'\nImbalance ratio: {ratio:.2f} (< 0.2 = serious imbalance)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=target, ax=axes[0], palette='Set2')
axes[0].set_title('Target Distribution')
axes[0].set_xticklabels(['Not Promoted', 'Promoted'])

sns.boxplot(data=df, x=target, y='salary_log', ax=axes[1], palette='Set2')
axes[1].set_title('Salary (log) vs Promoted')
axes[1].set_xticklabels(['Not Promoted', 'Promoted'])

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 9: All Features vs Target ────────────────────────────────────────────
feature_cols = ['age', 'salary_log', 'experience', 'score']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for i, col in enumerate(feature_cols):
    ax = axes.flatten()[i]
    sns.boxplot(data=df, x='promoted', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} vs promoted')
    ax.set_xticklabels(['Not Promoted', 'Promoted'])

plt.suptitle('Feature Distributions by Target Class', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 10: Categorical Column Analysis ──────────────────────────────────────
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    print(f'{col}: {df[col].nunique()} unique | {df[col].isnull().sum()} missing')
    print(df[col].value_counts().head(5))
    print()

fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))
if len(cat_cols) == 1:
    axes = [axes]
for i, col in enumerate(cat_cols):
    top = df[col].value_counts().head(8)
    axes[i].barh(top.index, top.values, color=f'C{i}', edgecolor='white')
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel('Count')
plt.suptitle('Categorical Column Distributions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 11: Pairplot — All Numeric Feature Relationships ─────────────────────
pairplot_cols = ['age', 'salary_log', 'experience', 'score', 'promoted']
sns.pairplot(
    df[pairplot_cols].dropna(),
    hue='promoted',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15}
)
plt.suptitle('Pairplot — Feature Relationships by Promotion Status', y=1.01)
plt.show()

In [ ]:
# ── CELL 12: Full EDA Summary Function ────────────────────────────────────────
def run_eda(df, target_col=None):
    print('=' * 55)
    print('STEP 1 — STRUCTURE')
    print('=' * 55)
    print(f'Shape: {df.shape}')
    print(f'\nData Types:\n{df.dtypes}')

    print('\n' + '=' * 55)
    print('STEP 2 — MISSING VALUES')
    print('=' * 55)
    missing = (df.isnull().mean() * 100).round(2)
    missing = missing[missing > 0].sort_values(ascending=False)
    print(missing if len(missing) > 0 else 'No missing values ✓')

    print('\n' + '=' * 55)
    print('STEP 3 — DUPLICATES')
    print('=' * 55)
    d = df.duplicated().sum()
    print(f'Duplicate rows: {d} ({d/len(df)*100:.1f}%)')

    print('\n' + '=' * 55)
    print('STEP 4 — NUMERIC SUMMARY + SKEWNESS')
    print('=' * 55)
    num_cols = df.select_dtypes(include=[np.number]).columns
    print(df[num_cols].describe().round(2))
    skew = df[num_cols].skew().sort_values(ascending=False)
    high = skew[abs(skew) > 1]
    if len(high) > 0:
        print(f'\n⚠ Highly skewed (log transform recommended): {high.index.tolist()}')

    print('\n' + '=' * 55)
    print('STEP 5 — CATEGORICAL COLUMNS')
    print('=' * 55)
    cat_cols = df.select_dtypes(include=['object','category']).columns
    for col in cat_cols:
        print(f'\n{col}: {df[col].nunique()} unique')
        print(df[col].value_counts().head(3).to_string())

    if target_col and target_col in df.columns:
        print('\n' + '=' * 55)
        print(f'STEP 6 — TARGET: {target_col}')
        print('=' * 55)
        vc = df[target_col].value_counts()
        print(vc)
        ratio = vc.min() / vc.max()
        flag = '⚠ IMBALANCED' if ratio < 0.3 else '✓ Balanced'
        print(f'Imbalance ratio: {ratio:.2f}  {flag}')

run_eda(df, target_col='promoted')

In [ ]:
# ── CELL 13: Final Cleaned Dataset Ready for Modeling ─────────────────────────
# Fill remaining missing values
df['salary']     = df.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median()))
df['salary_log'] = np.log1p(df['salary'])
df['score']      = df['score'].fillna(df['score'].median())
df['department'] = df['department'].fillna(df['department'].mode()[0])

# Encode categoricals
df['dept_code']   = df['department'].map({'Eng':1,'Sales':2,'HR':3,'Product':4,'Legal':5})
df['gender_code'] = df['gender'].map({'M':0,'F':1})

# Final feature matrix
feature_cols = ['age', 'salary_log', 'experience', 'score', 'dept_code', 'gender_code']
X = df[feature_cols].to_numpy()
y = df['promoted'].to_numpy()

print('Final feature matrix shape:', X.shape)
print('Target shape:', y.shape)
print('Missing values remaining:', pd.DataFrame(X).isnull().sum().sum())
print('\nReady for model training ✓')